# Real/Bogus Classification — CNN (ResNet-18)

Same dataset and preprocessing pipeline as the CvT notebook, but the backbone is swapped to **ResNet-18** via HuggingFace `transformers`.

In [1]:
from datasets import load_dataset

In [2]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
from transformers import (
    AutoImageProcessor,
    ResNetForImageClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Load Dataset

In [4]:
dataset = load_dataset("Santhosh2312/real-bogus-test_30k")
print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/394 [00:00<?, ?B/s]

train-00000-of-00012.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

train-00001-of-00012.parquet:   0%|          | 0.00/87.6M [00:00<?, ?B/s]

train-00002-of-00012.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

...

train-00009-of-00012.parquet:   0%|          | 0.00/94.0M [00:00<?, ?B/s]

train-00010-of-00012.parquet:   0%|          | 0.00/138M [00:00<?, ?B/s]

train-00011-of-00012.parquet:   0%|          | 0.00/541M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29067 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 29067
    })
})
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=950x362 at 0x7F083FCEB410>, 'label': 0}


## Model & Processor

We use `microsoft/resnet-18` — a lightweight but strong CNN baseline that works natively with the HuggingFace `Trainer`.

In [5]:
model_name = "microsoft/resnet-18"
processor = AutoImageProcessor.from_pretrained(model_name)

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

The image processor of type `ConvNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


## Data Preprocessing

Identical to the CvT pipeline:
1. Split each stitched 3-panel image horizontally.
2. Convert each panel to grayscale.
3. Run the ResNet processor (resize + normalise).
4. Keep one channel per panel and concatenate → `(3, H, W)` tensor.

In [6]:
def transform(example):
    images = example["image"]
    labels = example["label"]

    processed_images = []

    for img in images:
        # Split the stitched image into three panels
        width, height = img.size
        split_width = width // 3

        panel1 = img.crop((0,              0, split_width,   height)).convert("L")
        panel2 = img.crop((split_width,    0, 2*split_width, height)).convert("L")
        panel3 = img.crop((2*split_width,  0, width,         height)).convert("L")

        # Process each panel through the ResNet processor
        panel1 = processor(images=panel1, return_tensors="pt")["pixel_values"].squeeze(0)
        panel2 = processor(images=panel2, return_tensors="pt")["pixel_values"].squeeze(0)
        panel3 = processor(images=panel3, return_tensors="pt")["pixel_values"].squeeze(0)

        # Each panel: (3, H, W) with identical channels — keep one channel each
        panel1 = panel1[0:1, :, :]
        panel2 = panel2[0:1, :, :]
        panel3 = panel3[0:1, :, :]

        # Stack → (3, H, W) pseudo-RGB tensor
        combined = torch.cat([panel1, panel2, panel3], dim=0)
        processed_images.append(combined)

    return {
        "pixel_values": torch.stack(processed_images),
        "labels": torch.tensor(labels),
    }

## Train / Validation / Test Splits

In [7]:
full_dataset = dataset["train"]
labels = full_dataset["label"]

# Hold out 200 stratified samples for final test
train_indices, test_indices = train_test_split(
    np.arange(len(full_dataset)),
    test_size=200,
    stratify=labels,
    random_state=42,
)

train_dataset = full_dataset.select(train_indices)
test_dataset  = full_dataset.select(test_indices)

print("Train size:", len(train_dataset))
print("Test size :", len(test_dataset))

Train size: 28867
Test size : 200


In [8]:
labels_train = train_dataset["label"]

# 1 % of training data as validation
train_indices, val_indices = train_test_split(
    np.arange(len(train_dataset)),
    test_size=0.01,
    stratify=labels_train,
    random_state=42,
)

final_train_dataset = train_dataset.select(train_indices)
val_dataset         = train_dataset.select(val_indices)

print("Final train size :", len(final_train_dataset))
print("Validation size  :", len(val_dataset))

Final train size : 28578
Validation size  : 289


In [9]:
final_train_dataset = final_train_dataset.with_transform(transform)
val_dataset         = val_dataset.with_transform(transform)
test_dataset        = test_dataset.with_transform(transform)

## Build ResNet-18 Classifier

In [10]:
model = ResNetForImageClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "bogus", 1: "real"},
    label2id={"bogus": 0, "real": 1},
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                        
--------------------+----------+----------------------------------------------------------------------------------------
classifier.1.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])
classifier.1.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


## Training Arguments

In [11]:
training_args = TrainingArguments(
    output_dir="./results_cnn",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False,   # required — keeps pixel_values & labels
)

## Custom Collator & Trainer

In [12]:
def custom_data_collator(features):
    batch = {}
    batch["pixel_values"] = torch.stack([f["pixel_values"] for f in features])
    batch["labels"]       = torch.tensor([f["labels"]       for f in features])
    return batch

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_train_dataset,
    eval_dataset=val_dataset,
    data_collator=custom_data_collator,
)

## Train

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.066214,0.060026
2,0.025565,0.000698
3,0.021054,0.016689
4,0.000758,0.000415
5,0.002491,0.000123


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17865, training_loss=0.03548258033119978, metrics={'train_runtime': 2665.1992, 'train_samples_per_second': 53.613, 'train_steps_per_second': 6.703, 'total_flos': 1.4475095787061002e+18, 'train_loss': 0.03548258033119978, 'epoch': 5.0})

## Evaluate on Test Set

In [15]:
trainer.evaluate(test_dataset)

{'eval_loss': 0.028729097917675972,
 'eval_runtime': 2.7288,
 'eval_samples_per_second': 73.291,
 'eval_steps_per_second': 9.161,
 'epoch': 5.0}

In [16]:
# Detailed metrics
predictions = trainer.predict(test_dataset)

logits = predictions.predictions
true_labels = predictions.label_ids
preds = np.argmax(logits, axis=-1)

accuracy  = accuracy_score(true_labels, preds)
precision = precision_score(true_labels, preds)
recall    = recall_score(true_labels, preds)
f1        = f1_score(true_labels, preds)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.9950
Precision: 0.9907
Recall   : 1.0000
F1 Score : 0.9953


## Save Model

In [17]:
trainer.save_model("cnn_real_bogus_model")
processor.save_pretrained("cnn_real_bogus_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['cnn_real_bogus_model/preprocessor_config.json']

In [18]:
!zip -r cnn_real_bogus_model.zip cnn_real_bogus_model

  adding: cnn_real_bogus_model/ (stored 0%)
  adding: cnn_real_bogus_model/config.json (deflated 54%)
  adding: cnn_real_bogus_model/model.safetensors (deflated 7%)
  adding: cnn_real_bogus_model/training_args.bin (deflated 53%)
  adding: cnn_real_bogus_model/preprocessor_config.json (deflated 43%)


In [19]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [20]:
!cp cnn_real_bogus_model.zip /content/drive/MyDrive/cnn_real_bogus_model.zip
print("Saved to Google Drive.")

cp: cannot create regular file '/content/drive/MyDrive/cnn_real_bogus_model.zip': No such file or directory
Saved to Google Drive.
